[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Optimization/Manifold_Optimization.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization on Manifolds

When the constraint isn't a fence but a *surface* — unit spheres, orthonormal frames — penalties and projections fight the geometry. Riemannian optimization walks **along** the surface instead: two sessions, ending with an orthogonality-constrained eigenproblem solved natively and verified against `eigh`.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb), [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Riemannian Gradients & Retraction* (~40 min)
**Goal:** project the gradient onto the tangent space, step, retract — descent that never leaves the surface.
**Builds on:** [Optimization](./Optimization.ipynb) S2. &nbsp; **Feeds into:** Session 2 (the Stiefel manifold).

---

## 2. Walking on Curved Ground

💡 **Intuition.** On a sphere, the Euclidean gradient points *off* the surface — following it and re-normalizing is a fight. The Riemannian recipe makes peace with the geometry: (1) **project** the gradient onto the tangent plane (the directions you can actually move), (2) step, (3) **retract** back onto the manifold (for the sphere: normalize). All the [convergence theory](./Optimization.ipynb) carries over with Euclidean distance replaced by geodesic distance. On the sphere with $f = x^T S x$, the tangent-projected gradient is $2(Sx - (x^TSx)x)$ — zero exactly at **eigenvectors**: eigenproblems ARE Riemannian critical points (as [Lagrange already hinted](./Optimization.ipynb)).

In [ ]:
# Rayleigh-quotient minimization on the sphere — ORACLE: numpy's eigh

# YOUR CODE HERE


**What just happened — two runs of the identical algorithm, deliberately.** Both `riemannian_gd` calls run the exact same three-step recipe (tangent projection, step, retract) for 300 iterations. They land in completely different places:

| | `S` (Wishart) | `S1` (isolated min) |
|---|---|---|
| final Rayleigh quotient | 0.5015 | 1.00000000 |
| true smallest eigenvalue | 0.0097 | 1.00000000 |
| eigenvector alignment | — | 1.000000 |

`S`'s run stalls 52× away from its target — not because the algorithm is wrong, but because `S = M @ M.T` for a Gaussian $M$ is a Wishart matrix, and by [Marchenko–Pastur](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) its density behaves like $1/\sqrt{x}$ near zero: the two smallest eigenvalues sit almost on top of each other, so the convergence rate — governed by the relative gap $(\lambda_2-\lambda_1)/(\lambda_{\max}-\lambda_1)$ — is glacial. `S1` uses the identical code but a matrix built with one eigenvalue *deliberately isolated* from the rest (relative gap ≈ 0.21 instead of ≈ 0), and it converges to machine precision by iteration ~70, with the eigenvector alignment landing at exactly 1.000000. The plot shows this directly: the solid curve falls in a straight line on the log axis until it hits the noise floor of double precision (~1e-16, clipped in the plot so `semilogy` doesn't choke on the occasional roundoff-negative gap); the dashed curve never leaves its starting altitude.

**The lesson is sharper for having both runs in one cell.** Riemannian optimization's entire job is to guarantee you never leave the manifold — and it does that job perfectly in both runs; the sphere constraint is satisfied to machine precision throughout. What it does *not* do is fix conditioning: if the underlying objective has a near-degenerate spectral gap, projecting the gradient onto a tangent space and retracting doesn't change the convergence rate, because the rate is a property of the objective's curvature, not of the constraint. **Riemannian optimization fixes the constraint, not the conditioning** — the two are genuinely separate problems, and this cell solves only the first one for `S`, requiring a change to the matrix (not the algorithm) to solve the second.

Session 2 is the same lesson from the other side: it ascends toward the *largest* eigenvalues of a Wishart matrix, which are well separated, so the identical recipe converges cleanly there without needing an isolated-eigenvalue trick at all.

---
### 🕐 Session 2 of 2 — *The Stiefel Manifold: Orthonormal Frames* (~40 min)
**Goal:** optimize over ORTHONORMAL MATRICES; recover a subspace, verified against eigh.
**Builds on:** Session 1.

---

## 3. Frames That Stay Frames

💡 **Intuition.** Many problems want a whole orthonormal *frame* $X \in \mathbb{R}^{n \times k}$, $X^TX = I$ — PCA subspaces, [dictionary](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) atoms, [beamformer banks](../../Intro_DSP/Array_Processing.ipynb). That set is the **Stiefel manifold**. Same three-step dance: tangent projection $\xi = G - X\,\mathrm{sym}(X^TG)$, step, retract via the [QR factorization](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — Q *is* the nearest-frame map. No Lagrange multipliers, no drift, orthonormal to machine precision at every iterate.

In [ ]:
# top-k subspace by Stiefel gradient ASCENT on tr(XᵀSX) — ORACLE: eigh's top-k subspace
# subspace distance: principal angles via SVD of the cross-Gram

# YOUR CODE HERE


**What just happened.** Three results, and all three are exact:

- Orthonormality drift $\|X^\top X - I\| = 2.2\times10^{-16}$ — one unit in the last place of double precision, after **500 iterations**, with no accumulation whatsoever.
- Principal-angle cosines against `eigh`'s top-5 subspace: `[1. 1. 1. 1. 1.]`, with the `assert` requiring every one above 0.9999.
- Trace captured 620.8893 against an optimal 620.8893.

**The drift figure is the claim worth making loudly.** We enforced $k(k+1)/2 = 15$ scalar constraints, and after 500 steps they hold to machine precision — not approximately, not with a tuned penalty weight, and with no tendency to grow. A penalty method $\lambda\|X^\top X - I\|^2$ would need $\lambda$ tuned, would never be exactly feasible, and would drift or oscillate depending on the weight. Retraction gives exact feasibility *for free at every iterate*, which is why this matters in applications where a matrix must genuinely be a rotation — robotics, attitude estimation, unitary recurrent networks.

QR is what makes it clean: the `Q` factor is the **nearest orthonormal frame** to a given matrix, so "step off the manifold, then take Q" is a principled projection rather than a repair hack. And it is numerically excellent, which is why the drift is machine-epsilon rather than merely small.

**The principal-angle check is the right verification, and it is subtler than it looks.** We are recovering a *subspace*, not a specific basis — two different orthonormal bases can span the same space, so demanding $X = V_{\text{top}}$ would be wrong and would fail on a correct answer. The singular values of $X^\top V_{\text{top}}$ are the cosines of the principal angles between the two subspaces, and all five equalling 1.0 means the subspaces **coincide exactly**, whatever bases they happen to use. Testing the invariant rather than the representation is the general lesson.

**Now compare with Session 1, because the contrast is the most useful thing here — and this time both halves of it converged.** Same three-step recipe, same manifold family, and this cell lands exactly using `S`'s **top** 5 eigenvalues, which are well separated in a Wishart spectrum. Session 1's cell targets the *bottom* of that same spectrum with the identical recipe: run against the raw Wishart `S`, it stalls (the smallest eigenvalues are packed together); run against a matrix with one eigenvalue deliberately isolated, it converges to machine precision in ~70 iterations. The difference in every case is **spectral gap**, not geometry.

Having all three runs across the two sessions is more instructive than three clean successes would have been: **Riemannian optimisation guarantees feasibility and inherits conditioning.** Staying on the manifold is free; converging on it is exactly as hard as the underlying problem's spectral gap makes it.

**And one payoff worth naming.** Optimising over orthonormal matrices is how orthogonality-constrained RNNs eliminate the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) *analytically* — a unitary recurrence has every singular value equal to 1, so gradients neither decay nor explode by construction. That is this manifold solving, at the level of structure, a problem the RNN workshop could only mitigate with gates and clipping.

**Where this bites in practice:** orthogonality-regularized RNNs (unitary evolution kills the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) analytically), ICA's whitened rotations ([BSS workshop](../../Intro_DSP/ICA_Blind_Source_Separation.ipynb)), and low-rank matrix completion on fixed-rank manifolds.

## 4. Conclusion

Project to the tangent, step, retract: constrained optimization without constraints, converging to `eigh`'s answers (verified to 6 decimals) while staying orthonormal to machine precision. When your parameter *is* a geometry, optimize in it.

---
## Where next

- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — QR as the retraction workhorse.
- [Sparse & Dictionary Learning](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) — unit-norm atom constraints, everywhere.